In [1]:
import pandas as pd
import numpy as np
import random
from itertools import product

# 设置随机种子以确保可重复性
random.seed(42)
np.random.seed(42)

def create_negative_samples(positive_df, gene_list=None, allow_self_loop=False):
    """
    为基因调控网络构建负样本
    
    参数:
    positive_df: pandas DataFrame, 正样本数据，至少包含两列表示基因对
    gene_list: list, 所有基因的列表（可选）。如果为None，则从正样本中提取所有基因
    allow_self_loop: bool, 是否允许自环（基因对自身）。默认为False
    
    返回:
    negative_df: pandas DataFrame, 负样本数据
    """
    
    # 确保正样本数据有两列
    if len(positive_df.columns) < 2:
        raise ValueError("正样本DataFrame至少需要两列来表示基因对")
    
    # 获取基因对列名
    gene_col1, gene_col2 = positive_df.columns[:2]
    
    # 如果没有提供基因列表，从正样本中提取所有唯一基因
    if gene_list is None:
        tf_list = list(set(positive_df[gene_col1].tolist()))
        gene_list = list(set((positive_df[gene_col2].tolist())))

    # 创建所有可能的基因对（包括自环或不包括，根据参数）
    if allow_self_loop:
        all_possible_pairs = list(product(gene_list, repeat=2))
    else:
        all_possible_pairs = [(g1, g2) for g1 in tf_list for g2 in gene_list if g1 != g2]
    
    # 将正样本转换为集合以便快速查找
    positive_pairs = set(positive_df.apply(lambda row: tuple(row[:2]), axis=1))
    
    # 获取所有非正样本的基因对
    negative_candidates = [pair for pair in all_possible_pairs if pair not in positive_pairs]
    
    # 检查是否有足够的负样本候选
    n_positive = len(positive_df)
    if len(negative_candidates) < n_positive:
        print(f"警告: 负样本候选数量({len(negative_candidates)})少于正样本数量({n_positive})")
        print("将使用所有可用的负样本候选")
        n_samples = min(n_positive, len(negative_candidates))
    else:
        n_samples = n_positive
    
    # 随机抽取负样本
    selected_negative_pairs = random.sample(negative_candidates, n_samples)
    
    # 创建负样本DataFrame
    negative_df = pd.DataFrame(selected_negative_pairs, columns=[gene_col1, gene_col2])
    
    # 如果正样本有其他列，可以在负样本中添加对应的占位符列
    if len(positive_df.columns) > 2:
        for col in positive_df.columns[2:]:
            # 对于数值列，可以填充0；对于分类列，可以填充空值或特定值
            if pd.api.types.is_numeric_dtype(positive_df[col]):
                negative_df[col] = 0
            else:
                negative_df[col] = np.nan
    
    return negative_df

# 示例用法
if __name__ == "__main__":   
    
    pos = pd.read_csv("BL--network.csv")
    #del pos["Score"]
    print(pos)
    pos["label"] = 1
    pos.columns=["TF","Target","label"]
    # 方法1: 使用从正样本中提取的基因列表
    negative_df1 = create_negative_samples(pos)
    negative_df1["label"]=0
    negative_df1.columns=["TF","Target","label"]
    print(len(set(list(negative_df1["TF"]))))
    print(len(set(list(pos["TF"]))))
    print((set(list(pos["TF"]))))
    all=pd.concat([pos,negative_df1])   
    print(all)
    all.to_csv("AllPair.csv",index=False)

       Gene1  Gene2
0        AES   LEF1
1        AES   RND3
2        APC  DNMT1
3        APC   ODC1
4         AR   BTG2
...      ...    ...
4612    TP53  SYNE2
4613    TP53   TLE1
4614    TP53  TPD52
4615  TWIST1  CLDN7
4616  TWIST1   DLC1

[4617 rows x 2 columns]
292
292
{'GSC', 'SOS2', 'HLX', 'STAT2', 'ZBTB2', 'PITX2', 'LMO2', 'FOXH1', 'RFXANK', 'ESRRG', 'SMAD5', 'IRF1', 'MXD4', 'POLR1A', 'SNAPC4', 'NR0B1', 'DNMT3A', 'MAFG', 'GATA2', 'SIRT1', 'ANKZF1', 'BTG2', 'ETV4', 'FOXP1', 'MAML3', 'PATZ1', 'TRIP6', 'ATF6', 'FXR1', 'ILF3', 'SFPQ', 'SMAD2', 'SNRPD1', 'MDM4', 'TCF4', 'ELK3', 'DEK', 'NONO', 'BTF3', 'REL', 'SOX4', 'POLR2D', 'SKIL', 'CUX1', 'NR2C1', 'CBFA2T2', 'POLR2H', 'RUVBL2', 'PBX1', 'PRDM14', 'PLAGL1', 'GATA3', 'UPF2', 'HOXB3', 'PSIP1', 'PPARA', 'HIF1A', 'HMGA1', 'SATB1', 'NFKBIA', 'MCM5', 'KHSRP', 'SMARCC1', 'MECOM', 'UHRF1', 'ZNF160', 'CEBPZ', 'NANOG', 'RUNX1', 'ZIC2', 'KAT6A', 'NAA15', 'SFMBT1', 'SNRPB', 'SF1', 'SOX7', 'DNAJC2', 'EWSR1', 'AES', 'NCOA6', 'CTNNB1', 'MCM7', 'PAWR

In [2]:
import pandas as pd
a=pd.read_csv("ExpressionData.csv")
a

,ANKMY2,ERCC-00004,PKIG,RBMS3,RND3,TNFRSF21,CTNNB1,GYPB,CITED2,MCM4,...,CLU,BCAT2,MXI1,ATF6,PITX2,DUSP4,ALCAM,ECHDC1,ZNF764,ODZ2
0,5.291342,11.848240,5.435313,0.000000,8.874959,5.005212,10.595469,0.000000,0.000000,8.059676,...,5.856683,1.465333,5.718745,6.796136,0.000000,0.666385,2.898789,8.906366,0.000000,0.000000
1,3.810594,12.214217,4.714161,0.000000,9.700017,3.969049,10.921381,0.000000,1.824212,8.512563,...,7.205588,1.394478,0.000000,6.774401,0.000000,0.000000,1.783834,7.260263,0.885219,0.000000
2,0.000000,12.209793,5.351280,0.000000,8.506564,3.702854,10.549123,0.000000,0.000000,7.079198,...,6.689091,3.324283,4.411884,0.000000,0.000000,1.323498,1.323498,8.060598,0.000000,0.000000
3,5.117262,12.304050,3.332802,5.651564,8.098419,0.000000,9.407523,0.000000,0.000000,7.347010,...,4.049480,4.158240,0.000000,6.674978,0.000000,0.000000,0.000000,4.259372,2.745173,0.000000
4,0.000000,11.950298,3.415339,3.819880,9.142426,2.264657,10.103435,0.000000,0.757566,8.431746,...,6.764728,4.873619,2.851143,7.182425,0.000000,2.705997,0.000000,6.862156,0.703525,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
753,0.000000,13.168978,6.980502,6.730429,0.803826,0.000000,10.503271,9.392196,3.080188,4.377924,...,0.000000,6.060063,2.800289,5.871538,6.060063,2.452687,7.633792,3.773714,0.000000,7.473598
754,6.870142,13.126407,5.922956,4.668564,0.000000,6.000948,10.733202,8.994642,2.783067,8.689516,...,0.000000,4.345561,6.921058,5.285280,0.000000,7.704532,6.694674,7.910061,1.431734,4.084790
755,0.000000,13.029575,5.892868,6.723402,8.839241,6.161694,11.179546,8.337842,2.338259,7.592743,...,0.000000,2.552943,2.552943,4.464122,0.000000,0.000000,4.877667,0.000000,0.440430,0.000000
756,8.130179,13.210965,4.934580,1.510372,0.000000,6.266198,10.818403,9.162744,3.069588,4.806040,...,0.000000,2.710724,0.000000,8.934508,5.996849,6.629454,0.000000,5.503731,0.783285,0.000000


In [4]:
#转为大写
import pandas as pd
a=pd.read_csv("pathway_hESC.csv")
col = list(a.columns)
cols = [i.upper() for i in col]
a.columns=cols
a.to_csv("pathway_hESC.csv",index=False)

In [5]:
import pandas as pd
a=pd.read_csv("Target.csv")
column_names = list(a["Target"])
uppercase_names = [name.upper() for name in column_names]
a["Target"] = uppercase_names
a.to_csv("AllPair.csv",index=False)

KeyError: 'Target'

In [6]:
import pandas as pd
a=pd.read_csv("ExpressionData.csv")
column_names = list(a.columns)
uppercase_names = [name.upper() for name in column_names]
a.columns = uppercase_names
a.to_csv("ExpressionData.csv",index=False)